# TenderMind على Colab
Runtime > Change runtime type > **T4 GPU**، وبعدين شغّل الخلايا بالترتيب.

In [ ]:
%cd /content
!rm -rf TenderMind && git clone -b stage-5g-production-hardening https://github.com/Younes695/TenderMind.git
%cd /content/TenderMind
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-ara poppler-utils zstd > /dev/null
!pip -q install -r requirements.txt

In [ ]:
# build the website (React)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get -qq install -y nodejs > /dev/null
%cd /content/TenderMind/frontend
!npm ci --silent && npm run build --silent
%cd /content/TenderMind

In [ ]:
# local AI model on the Colab GPU
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True); time.sleep(5)
!ollama pull qwen2.5:3b

In [ ]:
# start TenderMind (processing ON - GPU is on Colab, not your laptop)
import os, subprocess, time
env = dict(os.environ, TENDERMIND_TWO_STAGE_LLM="1", TENDERMIND_WORKERS_ENABLED="1",
           TENDERMIND_MAX_AI_WORKERS="2", TENDERMIND_ESCALATION_MODEL="off")
subprocess.Popen("cd /content/TenderMind && nohup python -m uvicorn app.main:app --host 0.0.0.0 --port 8001 > /content/app.log 2>&1 &",
                 shell=True, env=env)
time.sleep(8)
!curl -s -o /dev/null -w "server: %{http_code}
" http://localhost:8001/

In [ ]:
# public link (temporary, works while this notebook is running)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import subprocess, time, re
subprocess.Popen("nohup cloudflared tunnel --url http://localhost:8001 > /content/tunnel.log 2>&1 &", shell=True)
time.sleep(10)
print(re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open('/content/tunnel.log').read())[:1])

In [ ]:
# logs if something fails
!tail -n 30 /content/app.log